# 06. Trusted Setup — SRS と鍵生成

**前提ファイル**: `network.compiled`, `settings.json`
（`04_compile.ipynb`, `03_calibrate.ipynb` まで実行済み）

In [ ]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [ ]:
import ezkl, os

compiled_model_path = 'network.compiled'
settings_path       = 'settings.json'
pk_path             = 'test.pk'
vk_path             = 'test.vk'

for f in [compiled_model_path, settings_path]:
    assert os.path.exists(f), f"{f} がありません。"

## Trusted Setup とは（RareSkills 2-8 との対応）

RareSkills で学んだ内容：
- 秘密のランダム値 `τ`（タウ）を使って公開パラメータを生成
- `τ` は生成後に**必ず破棄**（「毒のある秘密」）
- 破棄されれば、誰も偽の証明を作れない

EZKL での流れ：
```
1. get_srs    → 既製の SRS（他者が行った Trusted Setup の結果）をダウンロード
2. setup      → SRS + このモデルの回路 → PK（証明鍵）+ VK（検証鍵）を生成
```

| 鍵 | 誰が使う | 用途 |
|---|---|---|
| **PK**（Proving Key） | 証明者 | `prove` で証明を作るとき |
| **VK**（Verification Key） | 誰でも | `verify` で証明を確認するとき |

In [ ]:
# SRS のダウンロード（logrows のサイズに合わせた SRS を取得）
# 初回は数十MB のダウンロード
print("SRS をダウンロード中...")
ezkl.get_srs(settings_path)
print("完了")

In [ ]:
# PK と VK を生成
print("鍵を生成中...")
res = ezkl.setup(compiled_model_path, vk_path, pk_path)

assert res == True
print(f"VK（検証鍵）: {os.path.getsize(vk_path):,} bytes")
print(f"PK（証明鍵）: {os.path.getsize(pk_path):,} bytes")

## VK は公開・共有できる

VK は誰に渡しても安全です。PK は証明者だけが持ちます。

森林・カーボンクレジット文脈での使い方イメージ：
```
森林管理者（証明者）
    → 計測データ + モデル + PK → proof.json を生成
    → VK と proof.json を第三者機関に送る

第三者機関（検証者）
    → VK + proof.json → verify → True なら認証
    （計測データもモデルも見なくてよい）
```

---
次: `07_prove_verify.ipynb` で証明を生成し、検証します。